# SmartSort Browser Lab — Train GPU, đánh giá trong trình duyệt

1. Trên web Render: tải ảnh, tự gán nhãn, chia train/validation/test và xuất `SmartSort_Colab_Train.zip`.
2. Chọn **Runtime → Change runtime type → T4 GPU** rồi chạy từng ô dưới đây.
3. Upload gói chỉ train. Colab train classifier của EfficientNet-B0 với backbone ImageNet đóng băng, lưu tất cả epoch/cấu hình.
4. Tải `SmartSort_Browser_Models.zip`, quay lại **đúng phiên trên cùng trình duyệt**, nhập model.
5. Bấm validation trong trình duyệt, so sánh/chốt cấu hình rồi mới mở test và xuất báo cáo.

Colab không nhận validation/test và không tự chọn epoch. Web chọn epoch bằng validation. Backbone được chuyển thành ONNX và kiểm tra đặc trưng so với PyTorch trước khi xuất. Mỗi gói chứa backbone một lần và classifier mỗi epoch. Máy học sinh chạy suy luận CPU trong trình duyệt; train vẫn dùng GPU Colab. Sao lưu ZIP phiên trước khi đổi máy hoặc xóa dữ liệu trình duyệt. Không dùng notebook Docker cũ cho bản Browser Lab.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chọn T4 GPU trong Runtime trước khi train.'
print(torch.cuda.get_device_name(0))
%pip -q install 'onnx>=1.17,<2' 'onnxruntime>=1.20,<2'

In [ ]:
"""Train only on a SmartSort train package. Validation and test stay on the web server."""
import io
import json
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision import transforms as T
from PIL import Image, ImageOps


def npz_state(state):
    buffer = io.BytesIO()
    np.savez_compressed(buffer, **{k: v.detach().cpu().numpy() for k, v in state.items()})
    return buffer.getvalue()


def train_package(package_path, result_path='SmartSort_Colab_Models.zip', require_cuda=True):
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if require_cuda and dev.type != 'cuda':
        raise RuntimeError('Chọn Runtime → Change runtime type → T4 GPU, rồi chạy lại. Notebook không tự chuyển train sang CPU.')
    if dev.type == 'cpu':
        torch.set_num_threads(4)
    with zipfile.ZipFile(package_path) as package:
        metadata = json.loads(package.read('train.json'))
        assert metadata['kind'] == 'smartsort_train' and metadata['version'] == 1
        assert metadata['classes'] == ['metal', 'paper', 'plastic']
        assert not any(p.startswith(('validation/', 'test/')) for p in package.namelist())
        settings = metadata['settings']
        seed = metadata['seed']
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if dev.type == 'cuda':
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        weights = EfficientNet_B0_Weights.DEFAULT
        norm = weights.transforms()
        def letterbox(image):
            return ImageOps.pad(image.convert('RGB'), (224,224), method=Image.Resampling.BICUBIC, color='white')
        evaluation = T.Compose([T.Lambda(letterbox), T.ToTensor(), T.Normalize(norm.mean,norm.std)])
        augmentation = T.Compose([T.Lambda(letterbox), T.RandomHorizontalFlip(), T.RandomRotation(15,fill=255),
                                  T.ColorJitter(brightness=.15,contrast=.15,saturation=.1),T.ToTensor(),T.Normalize(norm.mean,norm.std)])
        class TrainImages(Dataset):
            def __init__(self, transform):
                self.transform = transform
            def __len__(self):
                return len(metadata['images'])
            def __getitem__(self,i):
                row = metadata['images'][i]
                with Image.open(io.BytesIO(package.read('train/'+row['id']+'.png'))) as image:
                    x = self.transform(image)
                return x, metadata['classes'].index(row['label'])
        model = efficientnet_b0(weights=weights).to(dev).eval()
        for p in model.features.parameters():
            p.requires_grad = False
        start = time.perf_counter()
        xs, ys = [], []
        with torch.inference_mode():
            for view in range(settings['views']):
                for x,y in DataLoader(TrainImages(evaluation if view == 0 else augmentation),batch_size=16,num_workers=0):
                    xs.append(model.avgpool(model.features(x.to(dev))).flatten(1).cpu())
                    ys.append(y)
                print(f'Train views {view+1}/{settings["views"]}; validation/test không có trong gói.',flush=True)
        features, labels = torch.cat(xs), torch.cat(ys)
        criterion = nn.CrossEntropyLoss()
        candidates = []
        with zipfile.ZipFile(result_path,'w',zipfile.ZIP_DEFLATED) as output:
            output.writestr('backbone.npz',npz_state(model.features.state_dict()))
            for index,config in enumerate(settings['candidates']):
                torch.manual_seed(seed)
                head = nn.Sequential(nn.Dropout(config['dropout']),nn.Linear(1280,3)).to(dev)
                optimizer = torch.optim.AdamW(head.parameters(),lr=config['lr'],weight_decay=config['weight_decay'])
                trial_id = f'trial_{index+1}'
                history = []
                dl = DataLoader(TensorDataset(features,labels),batch_size=16,shuffle=True)
                for epoch in range(1,settings['epochs']+1):
                    head.train()
                    loss_sum = correct = count = 0
                    for x,y in dl:
                        x,y = x.to(dev),y.to(dev)
                        optimizer.zero_grad(set_to_none=True)
                        logits = head(x)
                        loss = criterion(logits,y)
                        loss.backward()
                        optimizer.step()
                        loss_sum += loss.item()*len(y)
                        correct += int((logits.argmax(1)==y).sum())
                        count += len(y)
                    filename = f'heads/{trial_id}/epoch_{epoch:03d}.npz'
                    output.writestr(filename,npz_state(head.state_dict()))
                    history.append({'epoch':epoch,'file':filename,'train_loss':loss_sum/count,'train_accuracy':correct/count})
                    if epoch == 1 or epoch % 10 == 0:
                        print(f'{trial_id} epoch {epoch}: train loss {loss_sum/count:.4f}',flush=True)
                candidates.append({'id':trial_id,'config':config,'checkpoints':history})
            bundle = {'kind':'smartsort_colab_models','version':1,'model':'efficientnet_b0',
                      'classes':metadata['classes'],'request_id':metadata['request_id'],
                      'session_id':metadata['session_id'],'split_sha256':metadata['split_sha256'],
                      'settings':settings,'candidates':candidates,'training_seconds':time.perf_counter()-start,
                      'device':str(dev),'torch_version':str(torch.__version__),'torchvision_weights':str(weights),
                      'preprocessing':{'size':224,'fill':'white','mean':list(norm.mean),'std':list(norm.std)}}
            output.writestr('bundle.json',json.dumps(bundle,ensure_ascii=False,indent=2))
    print('Đã xuất',result_path,'— chưa chạy validation, chưa đọc test.',flush=True)
    return str(Path(result_path).resolve())


"""Portable Colab trainer: train-only ZIP -> ONNX backbone + JSON epoch heads."""
import tempfile
if __name__!='__main__':
    from .train_colab import *


def train_browser_package(package_path, result_path='SmartSort_Browser_Models.zip', require_cuda=True):
    # train_package, torch, np, nn, efficientnet_b0 are provided by train_colab.py.
    import onnx
    import onnxruntime as ort
    with tempfile.TemporaryDirectory(prefix='smartsort-browser-') as temporary:
        raw_path = str(Path(temporary)/'raw.zip')
        train_package(package_path,raw_path,require_cuda=require_cuda)
        with zipfile.ZipFile(raw_path) as raw:
            bundle=json.loads(raw.read('bundle.json'))
            model=efficientnet_b0(weights=None).cpu().eval()
            with np.load(io.BytesIO(raw.read('backbone.npz')),allow_pickle=False) as state:
                model.features.load_state_dict({key:torch.from_numpy(state[key].copy()) for key in state.files})
            backbone=nn.Sequential(model.features,model.avgpool,nn.Flatten(1)).eval()
            onnx_path=str(Path(temporary)/'backbone.onnx')
            torch.onnx.export(backbone,torch.zeros(1,3,224,224),onnx_path,
                              input_names=['image'],output_names=['features'],opset_version=17,
                              dynamo=False,external_data=False)
            onnx.checker.check_model(onnx.load(onnx_path))
            # Verify converted features against PyTorch before sending to students.
            rng=np.random.default_rng(42)
            sample=rng.normal(size=(1,3,224,224)).astype(np.float32)
            with torch.inference_mode():reference=backbone(torch.from_numpy(sample)).numpy()
            runtime=ort.InferenceSession(onnx_path,providers=['CPUExecutionProvider'])
            actual=runtime.run(None,{'image':sample})[0]
            np.testing.assert_allclose(actual,reference,rtol=1e-3,atol=1e-4)
            bundle['kind']='smartsort_browser_models'
            bundle['format']='onnx_features_json_heads_v1'
            bundle['onnx_max_abs_error']=float(np.abs(actual-reference).max())
            bundle['preprocessing']['resize']='pillow_bicubic_v1'
            with zipfile.ZipFile(result_path,'w',zipfile.ZIP_DEFLATED) as output:
                output.write(onnx_path,'backbone.onnx')
                for candidate in bundle['candidates']:
                    for checkpoint in candidate['checkpoints']:
                        old=checkpoint['file']
                        checkpoint['file']=old.replace('.npz','.json')
                        with np.load(io.BytesIO(raw.read(old)),allow_pickle=False) as state:
                            head={'weight':state['1.weight'].tolist(),'bias':state['1.bias'].tolist()}
                        output.writestr(checkpoint['file'],json.dumps(head,separators=(',',':'),allow_nan=False))
                output.writestr('bundle.json',json.dumps(bundle,ensure_ascii=False,allow_nan=False))
    print('Đã xuất',result_path,'— nhập vào web Browser Lab, chưa chạy validation/test.',flush=True)
    return str(Path(result_path).resolve())



In [ ]:
from google.colab import files
uploaded=files.upload()
packages=[p for p in uploaded if p.lower().endswith('.zip')]
assert len(packages)==1, 'Upload đúng một gói train.'
package_path=packages[0]
del uploaded

In [ ]:
result_path=train_browser_package(package_path,require_cuda=True)

In [ ]:
files.download(result_path)